# LOGAN: length-controlled separation by biome and tranche

**Draft caption.** Exact within-length-bin ROC AUC (from all-ORF histograms) for complete Prodigal+ against all complete Prodigal− ORFs. (a) By biome at 100–149 aa, ordered by 3Di entropy (n = Prodigal+ ORFs in the bin). (b) By tranche (pilot, phaseb1, phaseb2) at 100–149 and 150–199 aa. (c) 3Di entropy within each length bin, by biome. ORF length within a bin is shown as a reference.

**Outputs:** `../images/figLS1_biome_batch_separation.pdf` (vector) and `../images/figLS1_biome_batch_separation.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figLS1_biome_batch_separation.tsv.gz` (864 rows; sha256 `f272a047da37…`): exact AUC by biome / tranche and length bin (15b_biome_batch_figure.py). Source on Setonix (`$LOGAN_ROOT`): `qc/analysis_final/biome_batch_separation.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figLS1_biome_batch_separation.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
R = pd.read_csv(DATA / "figLS1_biome_batch_separation.tsv.gz", sep="\t")
LEN_LABELS = ["30-49", "50-74", "75-99", "100-149", "150-199", "200-299", "300-499", "500-999", ">=1000"]
FEATS = ["aa_len", "aa_entropy", "twelve_state_entropy", "three_di_entropy", "mi_3di_12st"]
LAB = {"aa_len": "ORF length (within bin)", "aa_entropy": "AA entropy", "twelve_state_entropy": "12-state entropy",
       "three_di_entropy": "3Di entropy", "mi_3di_12st": "3Di–12st MI"}
COL = {"aa_len": SLOTS[6], "aa_entropy": SLOTS[3], "twelve_state_entropy": SLOTS[2], "three_di_entropy": SLOTS[0],
       "mi_3di_12st": SLOTS[1]}
BATCHES = ["pilot", "phaseb1", "phaseb2"]
w = 0.16
fig = plt.figure(figsize=(19, 10.5))
gs = fig.add_gridspec(2, 2, width_ratios=[1.45, 1], height_ratios=[1, 1.05], hspace=0.42, wspace=0.15)
axA = fig.add_subplot(gs[0, :])
sb = R[(R.by == "biome") & (R.length_bin == "100-149")]
order = sb[sb.feature == "three_di_entropy"].sort_values("roc_auc").stratum.tolist()
x = np.arange(len(order))
for j, f in enumerate(FEATS):
    v = sb[sb.feature == f].set_index("stratum").reindex(order).roc_auc
    axA.bar(x + (j - 2) * w, v, width=w, color=COL[f], label=LAB[f])
npos = sb[sb.feature == "aa_len"].set_index("stratum").positives
axA.set_xticks(x, [f"{b.replace('_', ' ')}\n{npos[b] / 1e6:.1f} M" for b in order], fontsize=9.5)
axA.set_ylim(0.5, 1.0)
axA.set_ylabel("ROC AUC within 100–149 aa")
axA.legend(ncol=5, fontsize=10, loc="upper left")
panel(axA, "a", x=-0.03)
axB = fig.add_subplot(gs[1, 0])
groups = [(bt, lb) for lb in ["100-149", "150-199"] for bt in BATCHES]
xb = np.arange(len(groups))
sbt = R[R.by == "batch"].set_index(["stratum", "length_bin", "feature"])
for j, f in enumerate(FEATS):
    v = [sbt.loc[(bt, lb, f), "roc_auc"] for bt, lb in groups]
    axB.bar(xb + (j - 2) * w, v, width=w, color=COL[f])
axB.set_xticks(xb, [f"{bt}\n{lb.replace('-', '–')} aa" for bt, lb in groups], fontsize=9.5)
axB.set_ylim(0.5, 1.0)
axB.set_ylabel("ROC AUC within bin")
panel(axB, "b", x=-0.06)
axC = fig.add_subplot(gs[1, 1])
hb = R[(R.by == "biome") & (R.feature == "three_di_entropy") & ~R.length_bin.isin(["all", "stratified"])]
M = hb.pivot(index="stratum", columns="length_bin", values="roc_auc").reindex(
    index=order[::-1], columns=[c for c in LEN_LABELS if c in set(hb.length_bin)])
sns.heatmap(M, ax=axC, cmap="viridis", vmin=0.5, vmax=1.0, annot=True, fmt=".2f", annot_kws={"fontsize": 6.5},
            cbar_kws={"label": "ROC AUC within bin", "shrink": 0.8})
axC.set_yticklabels([b.replace("_", " ") for b in order[::-1]], rotation=0, fontsize=9)
axC.set_xlabel("ORF length bin (aa)")
axC.set_ylabel("")
panel(axC, "c", x=-0.25)
save(fig, "figLS1_biome_batch_separation")